# Week 10 + 11 — Model Training & Zero-Shot Tweet Evaluation
**Task 3 | Angsar Serikbekov**

**Week 10:** Train LR + SVM baselines on IMDb (binary: positive / negative)

**Week 11:** Evaluate zero-shot on tweets (3-class) using confidence threshold for neutral

---
## 0. Setup

In [ ]:
!git clone https://github.com/imrushka/ML_group_project
%cd ML_group_project

In [ ]:
%pip install -q datasets emoji pyarrow joblib

In [ ]:
from pathlib import Path

processed_files = list(Path('data/processed').glob('imdb_*.csv'))
if len(processed_files) >= 3:
    print(f'Processed data found ({len(processed_files)} files) — skipping pipeline.')
else:
    print('Running data pipeline...')
    %run src/data_collection.py
    %run src/data_cleaning.py

---
# WEEK 10 — Model Training on IMDb

## 1. Load & explore data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train_df = pd.read_csv('data/processed/imdb_train.csv')
val_df   = pd.read_csv('data/processed/imdb_val.csv')
test_df  = pd.read_csv('data/processed/imdb_test.csv')

print('Split sizes:')
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'  {name:5s}: {len(df):>6,} rows | {df["label_str"].value_counts().to_dict()}')

train_df['word_count'] = train_df['text_clean'].str.split().str.len()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['word_count'].hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set(title='Word count distribution (train)', xlabel='words', ylabel='count')
train_df['label_str'].value_counts().plot.bar(ax=axes[1], color=['#e74c3c', '#2ecc71'])
axes[1].set(title='Label distribution (train)')
plt.tight_layout()
plt.savefig('logs/eda_train.png', dpi=120)
plt.show()

## 2. TF-IDF vectorization

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

    SEED = 42
Path('models').mkdir(exist_ok=True)
Path('logs').mkdir(exist_ok=True)

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}

X_train_raw = train_df['text_clean'].fillna('')
X_val_raw   = val_df['text_clean'].fillna('')
X_test_raw  = test_df['text_clean'].fillna('')
y_train = train_df['label'].astype(int)
y_val   = val_df['label'].astype(int)
y_test  = test_df['label'].astype(int)

vec = TfidfVectorizer(
    ngram_range=(1, 2), max_features=50_000,
    sublinear_tf=True, min_df=3, strip_accents='unicode'
)
X_train = vec.fit_transform(X_train_raw)  # fit ONLY on train — no leakage
X_val   = vec.transform(X_val_raw)
X_test  = vec.transform(X_test_raw)

joblib.dump(vec, 'models/tfidf_vectorizer.joblib')
print(f'Vocabulary size : {len(vec.vocabulary_):,}')
print(f'X_train shape   : {X_train.shape}')

## 3. Logistic Regression

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

present_labels = sorted(y_train.unique())
label_names_binary = [ID2LABEL[i] for i in present_labels]

lr = LogisticRegression(
    C=1.0, max_iter=1000, solver='lbfgs',
    multi_class='multinomial', random_state=SEED, n_jobs=-1
)
t0 = time.time()
lr.fit(X_train, y_train)
print(f'LR trained in {time.time()-t0:.1f}s')
joblib.dump(lr, 'models/lr_model.joblib')

for split_name, X, y in [('val', X_val, y_val), ('test', X_test, y_test)]:
    print(f'\n── LR | {split_name} ──')
    print(classification_report(y, lr.predict(X), target_names=label_names_binary, zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, lr.predict(X_test), display_labels=label_names_binary, colorbar=False, ax=ax
)
ax.set_title('LR — IMDb Test')
plt.tight_layout()
plt.savefig('logs/lr_confusion_imdb.png', dpi=120)
plt.show()

## 4. Linear SVM

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1.0, max_iter=2000, random_state=SEED, dual='auto')
t0 = time.time()
svm.fit(X_train, y_train)
print(f'SVM trained in {time.time()-t0:.1f}s')
joblib.dump(svm, 'models/svm_model.joblib')

for split_name, X, y in [('val', X_val, y_val), ('test', X_test, y_test)]:
    print(f'\n── SVM | {split_name} ──')
    print(classification_report(y, svm.predict(X), target_names=label_names_binary, zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, svm.predict(X_test), display_labels=label_names_binary, colorbar=False, ax=ax
)
ax.set_title('SVM — IMDb Test')
plt.tight_layout()
plt.savefig('logs/svm_confusion_imdb.png', dpi=120)
plt.show()

## 5. IMDb results summary

In [ ]:
import json
from sklearn.metrics import accuracy_score, f1_score

imdb_results = {}
for model_name, model in [('LogisticRegression', lr), ('LinearSVC', svm)]:
    imdb_results[model_name] = {}
    for split_name, X, y in [('val', X_val, y_val), ('test', X_test, y_test)]:
        y_pred = model.predict(X)
        imdb_results[model_name][split_name] = {
            'accuracy': round(accuracy_score(y, y_pred), 4),
            'macro_f1': round(f1_score(y, y_pred, average='macro', zero_division=0), 4),
        }

rows = []
for m, splits in imdb_results.items():
    for s, metrics in splits.items():
        rows.append({'model': m, 'split': s, **metrics})
print(pd.DataFrame(rows).set_index(['model','split']).to_string())

# Bar chart
test_f1 = {m: imdb_results[m]['test']['macro_f1'] for m in imdb_results}
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(test_f1.keys(), test_f1.values(), color=['steelblue', 'coral'], width=0.4)
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.axhline(0.80, color='green', linestyle='--', linewidth=1, label='Target ≥0.80')
ax.set(title='IMDb Test — Macro-F1', ylabel='Macro-F1', ylim=(0, 1.05))
ax.legend()
plt.tight_layout()
plt.savefig('logs/imdb_comparison.png', dpi=120)
plt.show()

---
# WEEK 11 — Zero-Shot Evaluation on Tweets

We apply the IMDb-trained models directly to tweets **without any retraining**.

**The neutral problem:** models only know positive/negative.
We handle neutral with a confidence threshold:
- If the model is **not confident enough** (max probability < threshold) → predict **neutral**
- Otherwise → predict pos/neg as normal

> Note: LinearSVC does not output probabilities natively.
> We use `CalibratedClassifierCV` to add probability support.

## 6. Load tweet test data

In [ ]:
tweet_df = pd.read_csv('data/processed/tweet_final_test.csv')
print(f'Tweet test rows: {len(tweet_df):,}')
print(tweet_df['label_str'].value_counts())

X_tweet_raw = tweet_df['text_clean'].fillna('')
y_tweet     = tweet_df['label'].astype(int)   # 0=neg, 1=neutral, 2=pos

# Transform with the SAME vectorizer fitted on IMDb train
X_tweet = vec.transform(X_tweet_raw)
print(f'X_tweet shape: {X_tweet.shape}')

## 7. Calibrate SVM for probability output

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# Calibrate on validation set (not train — avoids overfitting)
svm_calibrated = CalibratedClassifierCV(svm, cv='prefit')
svm_calibrated.fit(X_val, y_val)

print('SVM calibrated. Both LR and SVM now output probabilities.')

## 8. Threshold-based neutral prediction

We test 3 threshold values to find the best one.
- **Low threshold (0.55):** assigns neutral more aggressively
- **Medium (0.65):** balanced
- **High (0.75):** only assigns neutral when very uncertain

In [ ]:
import numpy as np

NEUTRAL_LABEL = 1
THRESHOLDS = [0.55, 0.65, 0.75]

def predict_with_threshold(model, X, threshold: float) -> np.ndarray:
    """
    If max class probability < threshold → predict neutral (1).
    Otherwise predict the most likely class.
    """
    proba = model.predict_proba(X)
    preds = np.argmax(proba, axis=1)
    max_proba = proba.max(axis=1)
    # Map model class indices to our label schema
    classes = model.classes_
    preds_mapped = np.array([classes[p] for p in preds])
    # Apply threshold: low confidence → neutral
    preds_mapped[max_proba < threshold] = NEUTRAL_LABEL
    return preds_mapped

print('Threshold function ready.')

## 9. Zero-shot results across thresholds

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score, ConfusionMatrixDisplay

label_names_3class = ['negative', 'neutral', 'positive']
tweet_results = {}

for model_name, model in [('LogisticRegression', lr), ('LinearSVC', svm_calibrated)]:
    tweet_results[model_name] = {}
    print(f'\n{"="*60}')
    print(f'  {model_name} — Zero-Shot on Tweets')
    print(f'{"="*60}')

    for threshold in THRESHOLDS:
        y_pred = predict_with_threshold(model, X_tweet, threshold)
        acc      = accuracy_score(y_tweet, y_pred)
        macro_f1 = f1_score(y_tweet, y_pred, average='macro', zero_division=0)

        tweet_results[model_name][threshold] = {
            'accuracy': round(acc, 4),
            'macro_f1': round(macro_f1, 4),
        }

        print(f'\n  threshold={threshold} | Accuracy={acc:.4f} | Macro-F1={macro_f1:.4f}')
        print(classification_report(
            y_tweet, y_pred,
            target_names=label_names_3class,
            zero_division=0
        ))

## 10. Pick best threshold & plot confusion matrices

In [ ]:
# Find best threshold per model (highest Macro-F1)
best = {}
for model_name, thresholds in tweet_results.items():
    best_t = max(thresholds, key=lambda t: thresholds[t]['macro_f1'])
    best[model_name] = best_t
    print(f'{model_name}: best threshold = {best_t} | Macro-F1 = {thresholds[best_t]["macro_f1"]}')

# Confusion matrices for best thresholds
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (model_name, model) in zip(axes, [('LogisticRegression', lr), ('LinearSVC', svm_calibrated)]):
    best_t = best[model_name]
    y_pred = predict_with_threshold(model, X_tweet, best_t)
    ConfusionMatrixDisplay.from_predictions(
        y_tweet, y_pred,
        display_labels=label_names_3class,
        colorbar=False, ax=ax
    )
    ax.set_title(f'{model_name}\nTweets zero-shot (threshold={best_t})')
plt.tight_layout()
plt.savefig('logs/tweet_zeroshot_confusion.png', dpi=120)
plt.show()

## 11. Domain gap summary — IMDb vs Tweets

In [ ]:
# Side-by-side: IMDb test F1 vs Tweet zero-shot F1 (best threshold)
summary_rows = []
for model_name in ['LogisticRegression', 'LinearSVC']:
    imdb_f1  = imdb_results[model_name]['test']['macro_f1']
    best_t   = best[model_name]
    tweet_f1 = tweet_results[model_name][best_t]['macro_f1']
    drop     = round(imdb_f1 - tweet_f1, 4)
    summary_rows.append({
        'model': model_name,
        'IMDb F1 (binary)': imdb_f1,
        'Tweet F1 (zero-shot)': tweet_f1,
        'Drop': drop,
        'Best threshold': best_t,
    })

summary_df = pd.DataFrame(summary_rows).set_index('model')
print(summary_df.to_string())

# Visual
x = range(len(summary_rows))
width = 0.3
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - width/2 for i in x], summary_df['IMDb F1 (binary)'],  width=width, label='IMDb (binary)',      color='steelblue')
ax.bar([i + width/2 for i in x], summary_df['Tweet F1 (zero-shot)'], width=width, label='Tweets (zero-shot)', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(summary_df.index, rotation=10)
ax.axhline(0.60, color='green', linestyle='--', linewidth=1, label='Tweet target ≥0.60')
ax.set(title='Domain Gap: IMDb → Tweets', ylabel='Macro-F1', ylim=(0, 1.05))
ax.legend()
plt.tight_layout()
plt.savefig('logs/domain_gap.png', dpi=120)
plt.show()

print('\nThis drop is the baseline for Week 12 adaptation to recover.')

## 12. Save all metrics & push to GitHub

In [ ]:
all_metrics = {
    'imdb': imdb_results,
    'tweet_zeroshot': tweet_results,
    'best_thresholds': best,
    'domain_gap': {row['model']: row['Drop'] for row in summary_rows},
}
with open('logs/training_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print('Saved → logs/training_metrics.json')

In [ ]:
!git config user.email "your_email@example.com"
!git config user.name  "Angsar Serikbekov"
!git add models/ logs/
!git commit -m "Week 10+11: LR+SVM training + zero-shot tweet eval [Angsar]"
!git push

---
## Handoff to Week 12 (Domain Adaptation)

Files ready for Yerlan & Muhammadjon:

| File | Used for |
|---|---|
| `models/tfidf_vectorizer.joblib` | Transform tweet pool |
| `models/lr_model.joblib` | CORAL / self-training baseline |
| `models/svm_model.joblib` | CORAL / self-training baseline |
| `logs/training_metrics.json` | Baseline F1 numbers to beat |
| `logs/domain_gap.png` | For presentation slides |

**Proposal targets:**
- ≥ 0.80 Macro-F1 on IMDb ✓ (expected)
- ≥ 0.60 Macro-F1 on tweets before adaptation (check logs)
- ≥ +0.05 F1 improvement after adaptation (Week 12 goal)